## 09-statement-metrics



In [7]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_auc_score, precision_recall_curve

# 1. Загрузите файл classification.csv. В нем записаны истинные классы
# объектов выборки (колонка true) и ответы некоторого классифика-
# тора (колонка predicted).
df_class = pd.read_csv('classification.csv')
df_scores = pd.read_csv('scores.csv')


#  Таблица ошибок: TP, FP, FN, TN
y_true = df_class['true']
y_pred = df_class['pred']


In [8]:
#2. Заполните таблицу ошибок классификации

TP = sum((y_true == 1) & (y_pred == 1))
FP = sum((y_true == 0) & (y_pred == 1))
FN = sum((y_true == 1) & (y_pred == 0))
TN = sum((y_true == 0) & (y_pred == 0))


print(f"TP FP FN TN: {TP} {FP} {FN} {TN}")

TP FP FN TN: 43 34 59 64


In [9]:
#  3. Посчитайте основные метрики качества классификатора
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)


print(f"Accuracy Precision Recall F1: {accuracy:.3f} {precision:.3f} {recall:.3f} {f1:.3f}")


Accuracy Precision Recall F1: 0.535 0.558 0.422 0.480


#### 5. Посчитайте площадь под ROC-кривой для каждого классификатора. Какой классификатор имеет наибольшее значение метрики AUC-ROC (укажите название столбца с ответами этого классификатора)?

In [10]:
# AUC-ROC для четырёх классификаторов

y_true_scores = df_scores['true']
auc_results = {}

for col in ['score_logreg', 'score_svm', 'score_knn', 'score_tree']:
    auc = roc_auc_score(y_true_scores, df_scores[col])
    auc_results[col] = auc
    print(f"{col} AUC: {auc:.3f}")

best_auc_col = max(auc_results, key=auc_results.get)
print(f"Best AUC classifier: {best_auc_col}")






score_logreg AUC: 0.719
score_svm AUC: 0.709
score_knn AUC: 0.635
score_tree AUC: 0.692
Best AUC classifier: score_logreg


#### 6. Какой классификатор достигает наибольшей точности (Precision) при полноте (Recall) не менее 70% (укажите название столбца с ответами этого классификатора)? Какое значение точности при этом получается?

In [11]:
best_precision = -1
best_precision_col = None
best_precision_value = None

for col in ['score_logreg', 'score_svm', 'score_knn', 'score_tree']:
    # precision_recall_curve возвращает precision, recall, thresholds
    # thresholds содержит пороги, для которых precision и recall вычислены
    prec, rec, _ = precision_recall_curve(y_true_scores, df_scores[col])
    # отбираем точки, где recall >= 0.7
    mask = rec >= 0.7
    if np.any(mask):
        max_prec_at_rec = np.max(prec[mask])
        # округляем до двух знаков как в задании
        max_prec_at_rec_rounded = round(max_prec_at_rec, 2)
        if max_prec_at_rec > best_precision:
            best_precision = max_prec_at_rec
            best_precision_col = col
            best_precision_value = max_prec_at_rec_rounded

print(f"Best precision at recall >= 0.7: {best_precision_col} {best_precision_value:.2f}")

Best precision at recall >= 0.7: score_tree 0.65
